# PtyREX Bayesian optimisation — simple campaign runner

All workflow and display functions are in `ptyrex_bayesian_campaign.py`. This notebook contains only campaign configuration plus manual and supervised-automatic examples.

**Safety:** automatic mode has a countdown before each submission, stops at convergence, resumes persisted pending jobs, and honours `STOP_AFTER_CURRENT_ITERATION`. Jupyter cannot reliably trap the Escape key; use **Kernel → Interrupt** during the countdown, or create the stop file from a terminal/second notebook.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import importlib
import ptyrex_bayesian_campaign as campaign
importlib.reload(campaign)

from bayes_optimisation_helpers import (
    CrystalEvaluationConfig,
    ObjectiveMetric)

## Campaign configuration — edit this cell for each parameter/campaign

In [ ]:
#parameters
CONFIG_DIR = Path('/dls/science/groups/e02/Chris/Code/Jupyter_Notebooks/Bayes_ptycho_test_final')

metric_config = CrystalEvaluationConfig(
    q_search=(4.4, 6.2), angle_spacing_deg=60.0,
    radial_half_width=0.12, n_angles=1440, n_radial=11,
    min_distance_deg=8.0, family_tolerance_deg=12.0,
    position_tolerance=0.35, fit_half_width=0.7,
    validation_kwargs={'min_success_fraction': 0.5, 'min_sigma': 0.01,
                       'max_sigma': 1.0, 'max_relative_rmse': 0.3},
    objective_metric=ObjectiveMetric.SYMMETRY, minimise=True,
)

campaign.configure_campaign(
    CONFIG_DIR=CONFIG_DIR,
    MASTER_TEMPLATE=CONFIG_DIR/'single_slice_Bayes_master.json',
    WORK_DIR=CONFIG_DIR/'dR_crystal_bayes',
    OPTIMISED_PARAMETERS_FILE=CONFIG_DIR/'optimised_parameters.json',
    PARAMETER_NAME='dR', PARAMETER_UNIT='m',
    PARAMETER_MODE='paired', #'scalar' or 'paired'
    PARAMETER_PATH=('process','common','scan','dR'),
    PARAMETER_MIN=3e-11, PARAMETER_MAX=4.5e-11,
    MIN_PARAMETER_SPACING=0.01e-11, CONVERGENCE_SPACING=0.01e-11,
    MAX_PROPOSAL_ATTEMPTS=20,
    BATCH_SIZE=4, MAX_CONCURRENT=4, RANDOM_STATE=42,
    POLL_SECONDS=30, SACCT_RETRIES=6, SACCT_RETRY_SECONDS=10,
    DRY_RUN=False,
    SLURM_HOST='wilson', SLURM_PARTITION='cs05r',
    SLURM_TIME='00:30:00', SLURM_MEMORY='200G',
    SLURM_CONSTRAINT='NVIDIA_Blackwell',
    PTYREX_DIR=Path('/dls_sw/e02/software/PtyREX2026/PtyREX_cuda12'),
    metric_config=metric_config,
)

print(campaign.CAMPAIGN_STATE)

optimizer = campaign.load_or_create_optimizer()
campaign.current_state_summary()

In [ ]:
print(campaign.CAMPAIGN_STATE)

## Manual loop — run one cell at a time
Inspect the proposal and generated files before submission. `evaluate_completed_batch()` performs `optimizer.tell()` exactly once.

In [ ]:
history = campaign.review_last_iteration()
if campaign.CUMULATIVE_CSV.exists():
    best = campaign.show_campaign_optimum()

In [ ]:
proposal = campaign.propose_next_batch()  # inspect; no files/jobs yet

In [ ]:
batch = campaign.prepare_proposed_batch()  # inspect JSONs + SLURM script

In [ ]:
job_id = campaign.submit_prepared_batch()

In [ ]:
accounting = campaign.monitor_submitted_batch()

In [ ]:
results = campaign.evaluate_completed_batch()
history = campaign.review_last_iteration()
best = campaign.show_campaign_optimum()

In [ ]:
campaign.current_state_summary()

## Automated supervised example — up to N loops

The function displays `review_last_iteration()` and the current optimum dashboard each loop, shows proposed points, waits 30 seconds before submission, and stops on convergence.

**Break options**

1. Before submission
   - During the countdown use:
       Kernel → Interrupt
     or
       Ctrl+C

   Result:
       No jobs submitted.

2. After the current submitted batch
   - From another notebook:

       campaign.request_stop()

   - Or from a terminal:

       touch <WORK_DIR>/STOP_AFTER_CURRENT_ITERATION

   Result:
       Current SLURM jobs continue,
       the batch is evaluated,
       optimizer.tell() is executed,
       then the optimisation loop exits cleanly.

3. Automatic
   - The loop stops automatically when convergence is detected.


In [ ]:
doc = campaign.doctor()

In [ ]:
act= campaign.next_action()

In [ ]:
#if converged reset:
campaign.reset_proposal(
    allow_converged=True,
    reason=(
        "Reduced dR spacing threshold "
        "from previous value to 0.05e-11 m"
    ),
)
campaign.current_state_summary() # Campaign status should now say 'evaluated'

#and generate fresh proposal
proposal = campaign.propose_next_batch()
proposal


In [ ]:
# Clear an old stop request before starting intentionally.
campaign.clear_stop_request()

final_state = campaign.run_automated_loops(
    n_loops=3,
    countdown_seconds=30,
    show_dashboard=True,
    clear_between_iterations=False,
    radial_q_min=1.0,
    fourier_q_max=10.0,
)

## Request a safe stop from a terminal or second notebook

In [ ]:
campaign.request_stop()  # current submitted batch is still completed/evaluated safely

## Optional reconstruction inspection

In [ ]:
campaign.show_campaign_reconstruction(12)
# campaign.compare_campaign_reconstructions([12, 13, 14, 15])